# AI/R&D Assignment: Curve Fitting & Parameter Extraction

This notebook documents the experimental process for extracting the unknown variables $\theta$, $M$, and $X$ from the coordinate dataset `xy_data.csv`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# Load dataset
df = pd.read_csv('../xy_data.csv')
x_actual = df['x'].values
y_actual = df['y'].values
print(f"Loaded {len(x_actual)} points from xy_data.csv")

## Mathematical Proof: 2D Coordinate Rotation

The parametric equations are:
$$x(t) = t \cos(\theta) - e^{M|t|} \sin(0.3t)\sin(\theta) + X$$
$$y(t) = 42 + t \sin(\theta) + e^{M|t|} \sin(0.3t)\cos(\theta)$$

Shifting origins:
$$\tilde{x} = x - X$$
$$\tilde{y} = y - 42$$

Projecting via rotation by $\theta$ onto the longitudinal curve axis isolates $t$:
$$t = \tilde{x}\cos(\theta) + \tilde{y}\sin(\theta)$$

Projecting onto the transverse axis isolates the exponential/oscillatory component:
$$e^{M|t|}\sin(0.3t) = -\tilde{x}\sin(\theta) + \tilde{y}\cos(\theta)$$

In [ ]:
def objective(params):
    theta_deg, M_val, X_val = params
    theta_r = np.radians(theta_deg)
    
    # Longitudinal projection
    t_recovered = (x_actual - X_val) * np.cos(theta_r) + (y_actual - 42.0) * np.sin(theta_r)
    
    # Transverse residuals
    actual_z = -(x_actual - X_val) * np.sin(theta_r) + (y_actual - 42.0) * np.cos(theta_r)
    expected_z = np.exp(M_val * np.abs(t_recovered)) * np.sin(0.3 * t_recovered)
    
    return np.mean(np.abs(actual_z - expected_z))

# Perform grid search + local refinement
res = minimize(objective, (30.0, 0.03, 55.0), bounds=[(0.0, 50.0), (-0.05, 0.05), (0.0, 100.0)], method='L-BFGS-B')
print("Optimization Results:")
print(f"Theta: {res.x[0]:.6f} degrees")
print(f"M: {res.x[1]:.6f}")
print(f"X: {res.x[2]:.6f}")